# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ihSomik/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Decision:** which pages should an SEO/content specialist review first? **Question:** can a model rank pages by *later* missed clicks (CTR below the norm for the page's position) better than a transparent rule, on clients it has never seen? Wrong-call cost: a false alarm wastes editor time; a miss leaves clicks unclaimed.

In [1]:
import json, os
import pandas as pd
print('Lane: CTR-fix review queue (scoring and ranking). Success = beat the Week-4 rule on held-out clients, same split, same metrics.')

Lane: CTR-fix review queue (scoring and ranking). Success = beat the Week-4 rule on held-out clients, same split, same metrics.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Warehouse: daily page-performance fact table joined to a content dimension (hashed ids only). Window: March 2026 (features days 1-21, label days 22-31). Kept Search Console rows; excluded paid, AI-referral, same-day outcome columns and client ids as features. Eligible pages: >=100 impressions and average position 1-20.

In [2]:
import json, os
import pandas as pd
data=pd.DataFrame({'item':['March 2026 daily rows','Rows with Search Console data','Pages with impressions (days 1-21)','Eligible pages','Clients among eligible','Share of impressions held by eligible pages'],
 'value':['9,841,378','36.7%','162,301','68,868 (42.4%)','38','76%']})
data

,item,value
0,March 2026 daily rows,"9,841,378"
1,Rows with Search Console data,36.7%
2,Pages with impressions (days 1-21),"162,301"
3,Eligible pages,"68,868 (42.4%)"
4,Clients among eligible,38
5,Share of impressions held by eligible pages,76%


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Baseline: score = max(expected CTR for position band - actual CTR, 0) x impressions. Label: later missed clicks in the top quarter of *training* pages. Features: 16 inputs known by day 21. Validation: 5-fold GroupKFold by client, identical folds for every method. Leakage: name/window guards, noise control, label-derived trap, train/test overlap, pooled-norm check.

In [3]:
import json, os
import pandas as pd
leak=pd.DataFrame([('Banned input names','PASS'),('Feature window ends before label window','PASS'),('Random-noise control (AUC 0.509)','PASS'),('No single feature above 0.85 AUC','PASS'),('Trap column inflates AUC (0.905 -> 1.000)','PASS'),('No client in both train and test','PASS'),('Pooled norm shift (0.785 vs 0.786)','PASS')],columns=['check','status'])
print('PASS = nothing found in this sample; it does not prove leakage is absent.')
leak

PASS = nothing found in this sample; it does not prove leakage is absent.


,check,status
0,Banned input names,PASS
1,Feature window ends before label window,PASS
2,Random-noise control (AUC 0.509),PASS
3,No single feature above 0.85 AUC,PASS
4,Trap column inflates AUC (0.905 -> 1.000),PASS
5,No client in both train and test,PASS
6,Pooled norm shift (0.785 vs 0.786),PASS


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Model vs baseline, same held-out client folds. The model ranks the whole list better (AUC) but did **not** capture more later missed clicks in the top-5% queue. Impressions alone reach AUC 0.837.

In [4]:
import json, os
import pandas as pd
res=pd.DataFrame([('Week-4 rule (baseline)',.785,.668,.904,.388),('Impressions only',.837,.552,.635,.338),('Logistic Regression',.876,.687,.830,.398),('Random Forest (shallow)',.891,.726,.868,.338),('Gradient Boosting (small)',.906,.761,.914,.374)],columns=['method','AUC','AvgPrecision','Precision@top5%','Captured@top5%']).set_index('method')
print('Paired AUC gain GB vs rule: +0.121, better in 5/5 folds. Captured gain: -0.014, better in 2/5.')
res

Paired AUC gain GB vs rule: +0.121, better in 5/5 folds. Captured gain: -0.014, better in 2/5.


,AUC,AvgPrecision,Precision@top5%,Captured@top5%
method,,,,
Week-4 rule (baseline),0.785,0.668,0.904,0.388
Impressions only,0.837,0.552,0.635,0.338
Logistic Regression,0.876,0.687,0.830,0.398
Random Forest (shallow),0.891,0.726,0.868,0.338
Gradient Boosting (small),0.906,0.761,0.914,0.374


## 5. Limitations

*What this work cannot claim.*

Cannot claim: causal effect of any rewrite or refresh; future clicks; anything outside March 2026, Search Console pages or unseen months. The label is a proxy against an in-sample pooled norm; page-level data cannot see query intent (branded queries, answer boxes, AI answers); the best model was chosen on the folds it is scored on. Directional decision-support only.

In [5]:
import json, os
import pandas as pd
limits=['One month, one label definition','Proxy label, not a measured rewrite outcome','Page-level, not query-level','Observational: no causal claim','Effort hours are assumptions']
print('\n'.join('- '+l for l in limits))

- One month, one label definition
- Proxy label, not a measured rewrite outcome
- Page-level, not query-level
- Observational: no causal claim
- Effort hours are assumptions


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Ranked by estimated clicks at stake per effort hour (upper bound, assumed effort). Every row needs human review; none is safe to automate. Fading pages rebounded 1.56x in the next 10 days, so verify before refreshing.

In [6]:
import json, os
import pandas as pd
recs=pd.DataFrame([(1,'Review title/description (page 1, CTR below norm)',20838,165331,0.5),(2,'Verify, then refresh fading pages',5031,44646,3.0),(3,'Improve relevance and links (positions 11-20)',7291,30553,1.5)],columns=['rank','action','pages','clicks_at_stake_30d','effort_h_assumed'])
recs['per_effort_hour']=(recs.clicks_at_stake_30d/(recs.pages*recs.effort_h_assumed)).round(1)
recs

,rank,action,pages,clicks_at_stake_30d,effort_h_assumed,per_effort_hour
0,1,"Review title/description (page 1, CTR below norm)",20838,165331,0.5,15.9
1,2,"Verify, then refresh fading pages",5031,44646,3.0,3.0
2,3,Improve relevance and links (positions 11-20),7291,30553,1.5,2.8


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Charts and tables on the deployed page: AUC and captured-clicks bars, fading-page decay bars, action-mix bars, model-vs-baseline table, recommendations table. Figures from Week 7 are in `work/figures/`; receipts are in `work/outputs/*.json`.

In [7]:
import json, os
import pandas as pd
found=[f for f in ['work/outputs/baseline_metrics.json','work/outputs/model_metrics.json','work/outputs/validation_audit.json','work/outputs/playbook_metrics.json'] if os.path.exists(f)]
print('Receipts found in this checkout:', found or 'none (run Weeks 4-7 first)')

Receipts found in this checkout: none (run Weeks 4-7 first)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.